# Thermal person detector: training and evaluation (Google Colab, Tesla T4)

Fine-tunes **YOLOv8n** (COCO-pretrained) on **8,778 real thermal images** (Hugging Face `Voxel51/Thermal-Person-Detector`, 18,695 person boxes), split 70/15/15 with seed 42.

| Cell | What it does |
|---|---|
| 1 | Loads the dataset, splits it, exports it to YOLO format, trains 40 epochs (results saved to Google Drive) |
| 2 | Test-set metrics, baseline (YOLOv8n without thermal training), size, CPU speed, 5 example detections (same code as `ai/evaluate.py`) |
| 3 | Confusion matrix at the drone's threshold, confidence 0.5 (same code as `ai/confusion_at_0.5.py`) |

Results and the full report: [`REPORT.md`](REPORT.md). The outputs below are the real run.

In [2]:
!pip install -q fiftyone ultralytics
from google.colab import drive
drive.mount('/content/drive')                      # save results outside Colab

import shutil
import fiftyone as fo
import fiftyone.utils.huggingface as fouh
import fiftyone.utils.random as four
from ultralytics import YOLO

# 1. Load the dataset (reuse it if it's already loaded: no more "name not available")
NAME = "Voxel51/Thermal-Person-Detector"
ds = fo.load_dataset(NAME) if fo.dataset_exists(NAME) else fouh.load_from_hub(NAME)
print(ds)
print("labels:", ds.count_values("ground_truth.detections.label"))

# 2. Fresh 70/15/15 split (remove old tags first)
ds.untag_samples(["train", "val", "test"])
four.random_split(ds, {"train": 0.70, "val": 0.15, "test": 0.15}, seed=42)
print("split:", ds.count_sample_tags())

# 3. Export to YOLO format into an EMPTY folder (no duplicates)
OUT = "/content/thermal_yolo"
shutil.rmtree(OUT, ignore_errors=True)
for split in ["train", "val", "test"]:
    ds.match_tags(split).export(
        export_dir=OUT,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        split=split,
        classes=["person"],
    )

# 4. Train, saving every epoch to Drive
model = YOLO("yolov8n.pt")
model.train(
    data=f"{OUT}/dataset.yaml",
    epochs=40, imgsz=640, batch=16,
    patience=10,          # stop early if no progress for 10 epochs
    seed=42,
    project="/content/drive/MyDrive/aeroday",
    name="thermal_person",
    exist_ok=True,
)

Mounted at /content/drive
Name:        Voxel51/Thermal-Person-Detector
Media type:  image
Num samples: 8778
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
labels: {'person': 18695}
split: {'test': 1317, 'train': 6145, 'val': 1316}
 100% |███████████████| 6145/6145 [7.7s elapsed, 0s remaining, 547.1 samples/s]       


INFO:eta.core.utils: 100% |███████████████| 6145/6145 [7.7s elapsed, 0s remaining, 547.1 samples/s]       


Directory '/content/thermal_yolo' already exists; export will be merged with existing files


 100% |███████████████| 1316/1316 [2.1s elapsed, 0s remaining, 578.0 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 1316/1316 [2.1s elapsed, 0s remaining, 578.0 samples/s]      


Directory '/content/thermal_yolo' already exists; export will be merged with existing files


 100% |███████████████| 1317/1317 [2.1s elapsed, 0s remaining, 574.2 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 1317/1317 [2.1s elapsed, 0s remaining, 574.2 samples/s]      


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/thermal_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=thermal_person, nbs=64, nms=None, opset=None, optimize=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78ac8153d320>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [3]:
# Evaluation of the thermal person detector. Paste this whole file into ONE new Colab cell
# and run it after training has finished. It writes everything the report needs into
#   /content/drive/MyDrive/aeroday/thermal_person/report/
# and downloads it as report.zip.
#
# What it does:
#   1. Test-set metrics of our model (images never used in training or model selection)
#   2. The same test for the original YOLOv8n (trained on normal photos, never on thermal)
#      -> shows what our fine-tuning adds
#   3. Model size and CPU speed (is it light enough for a drone's onboard computer?)
#   4. 5 example detections on real thermal test images
#   5. Everything saved to metrics.json + images, zipped and downloaded

import glob
import json
import os
import random
import shutil
import time

from ultralytics import YOLO

RUN = "/content/drive/MyDrive/aeroday/thermal_person"   # where training saved its results
DATA_DIR = "/content/thermal_yolo"
DATA = f"{DATA_DIR}/dataset.yaml"
OUT = f"{RUN}/report"
os.makedirs(f"{OUT}/examples", exist_ok=True)

# --- 0. If Colab was restarted, the exported dataset is gone: rebuild it (same seed = same split)
if not os.path.exists(DATA):
    import fiftyone as fo
    import fiftyone.utils.huggingface as fouh
    import fiftyone.utils.random as four
    NAME = "Voxel51/Thermal-Person-Detector"
    ds = fo.load_dataset(NAME) if fo.dataset_exists(NAME) else fouh.load_from_hub(NAME)
    ds.untag_samples(["train", "val", "test"])
    four.random_split(ds, {"train": 0.70, "val": 0.15, "test": 0.15}, seed=42)
    for split in ["train", "val", "test"]:
        ds.match_tags(split).export(export_dir=DATA_DIR, dataset_type=fo.types.YOLOv5Dataset,
                                    label_field="ground_truth", split=split, classes=["person"])

model = YOLO(f"{RUN}/weights/best.pt")


def box_metrics(m):
    return {"precision": round(float(m.box.mp), 4),
            "recall": round(float(m.box.mr), 4),
            "mAP50": round(float(m.box.map50), 4),
            "mAP50_95": round(float(m.box.map), 4),
            "speed_ms_per_image_gpu": {k: round(v, 2) for k, v in m.speed.items()}}


# --- 1. Our model on the TEST set (also draws the confusion matrix and the PR / F1 curves)
ours = model.val(data=DATA, split="test", project=RUN, name="test_eval", exist_ok=True, plots=True)
ours_m = box_metrics(ours)
print("OUR MODEL (test):", ours_m)

# --- 2. Baseline: the original YOLOv8n, never trained on thermal images ("person" = COCO class 0)
try:
    base = YOLO("yolov8n.pt").val(data=DATA, split="test", classes=[0], project=RUN,
                                  name="baseline_eval", exist_ok=True, plots=False)
    base_m = box_metrics(base)
except Exception as e:  # never block the report on the baseline
    base_m = {"error": str(e)}
print("BASELINE yolov8n, no thermal training (test):", base_m)

# --- 3. Size and CPU speed
from ultralytics.utils.torch_utils import get_flops
params = sum(p.numel() for p in model.model.parameters())
gflops = get_flops(model.model, 640)
layers = len(list(model.model.modules()))
test_imgs = sorted(glob.glob(f"{DATA_DIR}/images/test/*"))
sample = random.Random(0).sample(test_imgs, 50)
model.predict(sample[0], device="cpu", verbose=False)              # warm-up
t = time.time()
for p in sample:
    model.predict(p, device="cpu", imgsz=640, verbose=False)
cpu_ms = (time.time() - t) / len(sample) * 1000
print(f"Size: {params / 1e6:.2f} M parameters, {gflops:.1f} GFLOPs | CPU: {cpu_ms:.0f} ms/image = {1000 / cpu_ms:.1f} fps")

# --- 4. Five example detections on test images that contain people
with_people = [p for p in test_imgs
               if os.path.getsize(p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt") > 0]
for i, p in enumerate(random.Random(1).sample(with_people, 5), 1):
    r = model.predict(p, conf=0.5, verbose=False)[0]
    r.save(filename=f"{OUT}/examples/example_{i}.jpg")

# --- 5. Save the numbers, copy the plots, zip and download
metrics = {
    "dataset": {"name": "Voxel51/Thermal-Person-Detector (Hugging Face)", "images": 8778,
                "person_boxes": 18695, "split": {"train": 6145, "val": 1316, "test": 1317},
                "split_method": "random 70/15/15, seed 42"},
    "training": {"base_model": "yolov8n.pt (COCO pretrained)", "epochs_max": 40, "patience": 10,
                 "imgsz": 640, "batch": 16, "seed": 42},
    "test_our_model": ours_m,
    "test_baseline_yolov8n_no_thermal_training": base_m,
    "model_size": {"layers": int(layers), "parameters_M": round(params / 1e6, 2),
                   "GFLOPs": round(float(gflops), 1),
                   "weights_MB": round(os.path.getsize(f"{RUN}/weights/best.pt") / 1e6, 1)},
    "cpu_speed": {"ms_per_image": round(cpu_ms, 1), "fps": round(1000 / cpu_ms, 1),
                  "note": "Colab CPU, imgsz 640, 50 test images"},
}
with open(f"{OUT}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

for png in glob.glob(f"{RUN}/test_eval/*.png") + glob.glob(f"{RUN}/*.png"):  # confusion matrix, curves, results.png
    shutil.copy(png, OUT)
shutil.copy(f"{RUN}/results.csv", OUT)            # loss and metrics per epoch
shutil.copy(f"{RUN}/weights/best.pt", OUT)        # the trained model

print(json.dumps(metrics, indent=2))
shutil.make_archive("/content/report", "zip", OUT)
from google.colab import files
files.download("/content/report.zip")


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 788.4±330.7 MB/s, size: 27.1 KB)
val: Scanning /content/thermal_yolo/labels/test... 1317 images, 97 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1317/1317 1.9Kit/s 0.7s
val: New cache created: /content/thermal_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 83/83 5.6it/s 14.8s
                   all       1317       2872      0.921      0.854      0.938      0.672
Speed: 1.6ms preprocess, 4.2ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/drive/MyDrive/aeroday/thermal_person/test_eval
OUR MODEL (test): {'precision': 0.9206, 'recall': 0.8538, 'mAP50': 0.9379, 'mAP50_95': 0.6718, 'speed_ms_per_image_gpu': {'preprocess': 1.55, 'inference': 4.2, 'loss': 0.0, 'postproces

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# Confusion matrix at the drone's real operating threshold (conf >= 0.5).
# Paste into ONE new Colab cell after evaluate.py.
#
# Why: Ultralytics draws its default confusion matrix with every guess, down to 0.1 %
# confidence (needed to compute mAP), so it shows ~20,000 "false persons" that the drone
# would never report. The drone only reports a person at conf >= 0.5, so this is the
# matrix that describes its real behaviour.
import glob
import json
import shutil

from google.colab import files
from ultralytics.models.yolo.detect import DetectionValidator

RUN = "/content/drive/MyDrive/aeroday/thermal_person"
OUT = f"{RUN}/report"

v = DetectionValidator(args=dict(model=f"{RUN}/weights/best.pt", data="/content/thermal_yolo/dataset.yaml",
                                 split="test", conf=0.5, plots=True,
                                 project=RUN, name="test_conf05", exist_ok=True))
v()
m = v.confusion_matrix.matrix          # rows = predicted, columns = true; last = background
tp, fp, fn = int(m[0, 0]), int(m[0, -1]), int(m[-1, 0])
result = {"conf_threshold": 0.5, "true_positives": tp, "false_positives": fp, "false_negatives": fn,
          "precision": round(tp / (tp + fp), 4), "recall": round(tp / (tp + fn), 4)}
print(json.dumps(result, indent=2))

with open(f"{OUT}/confusion_at_0.5.json", "w") as f:
    json.dump(result, f, indent=2)
for png in glob.glob(f"{v.save_dir}/confusion_matrix*.png"):
    dst = f"{OUT}/" + png.split("/")[-1].replace("confusion_matrix", "confusion_matrix_conf0.5")
    shutil.copy(png, dst)
    files.download(dst)
files.download(f"{OUT}/confusion_at_0.5.json")


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1042.7±561.3 MB/s, size: 39.3 KB)
val: Scanning /content/thermal_yolo/labels/test.cache... 1317 images, 97 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1317/1317 115.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 83/83 3.7it/s 22.2s
                   all       1317       2872      0.941      0.829      0.818      0.595
Speed: 2.1ms preprocess, 4.3ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to /content/drive/MyDrive/aeroday/thermal_person/test_conf05
{
  "conf_threshold": 0.5,
  "true_positives": 2382,
  "false_positives": 149,
  "false_negatives": 490,
  "precision": 0.9411,
  "recall": 0.8294
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>